In [2]:
# 导入必要的库
import metpy.calc as mpcalc
from metpy.units import units
import gc  # 垃圾回收
import os
import shutil
from pathlib import Path
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
# Set data path
data_path = Path('/work/mh1498/m301257/processed_data/')


In [3]:
try:
    rhor_cntl_half = xr.open_dataarray('../3D_data/cntl/rho_all_levels.nc', chunks={'time': 10}).transpose('time', 'level', 'lat', 'lon')
    rhor_p4k_half = xr.open_dataarray('../3D_data/p4k/rho_all_levels.nc', chunks={'time': 10}).transpose('time', 'level', 'lat', 'lon')
    rhor_4co2_half = xr.open_dataarray('../3D_data/4co2/rho_all_levels.nc', chunks={'time': 10}).transpose('time', 'level', 'lat', 'lon')
    print(rhor_cntl_half.shape,rhor_p4k_half.shape,rhor_4co2_half.shape)
    print(rhor_cntl_half.dims)
except Exception as e:
    print(f"  警告: 未找到气压数据 ({e})")
    print("  将使用默认设置")
   

(5114, 22, 17, 180) (5114, 22, 17, 180) (5114, 22, 17, 180)
('time', 'level', 'lat', 'lon')


In [ ]:
rhor_cntl_half

<xarray.DataArray 'rho' (time: 5114, level: 22, lat: 17, lon: 180)> Size: 3GB
dask.array<transpose, shape=(5114, 22, 17, 180), dtype=float64, chunksize=(10, 22, 17, 180), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 41kB 1980-01-01 1980-01-02 ... 1993-12-31
  * level    (level) int64 176B 31 35 38 41 46 51 55 58 ... 81 83 84 85 87 89 90
  * lat      (lat) float64 136B -16.0 -14.0 -12.0 -10.0 ... 10.0 12.0 14.0 16.0
  * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0

: 

In [ ]:
def interpolate_half_to_full_level(rho_half):
    """
    将ICON模式half level上的密度插值到full level
    
    根据ICON垂直层次结构：
    - Half levels: k-1/2, k+1/2 包围着layer k
    - Full level k 位于两个half level中间
    - 使用线性插值: rho_full[k] = (rho_half[k-1/2] + rho_half[k+1/2]) / 2
    
    参数:
        rho_half: xarray.DataArray, shape (time, level, lat, lon)
                 Half level上的密度数据，level维度为num_lev+1
    
    返回:
        rho_full: xarray.DataArray, shape (time, level, lat, lon)
                 Full level上的密度数据，level维度为num_lev
    """
    print(f"输入数据形状: {rho_half.shape}")
    print(f"输入数据level范围: {rho_half.level.values.min()} - {rho_half.level.values.max()}")
    
    # 获取维度信息
    num_half_levels = rho_half.sizes['level']  # num_lev + 1
    num_full_levels = num_half_levels - 1       # num_lev
    
    # 对于full level k (k=1,...,num_lev):
    # rho_full[k] = (rho_half[k-1/2] + rho_half[k+1/2]) / 2
    # 即: rho_full[k-1] = (rho_half[k-1] + rho_half[k]) / 2  (索引从0开始)
    
    # 使用dask进行高效计算，避免立即加载所有数据到内存
    rho_half_lower = rho_half.isel(level=slice(0, -1))
    rho_half_upper = rho_half.isel(level=slice(1, None))
    
    # 计算平均值
    rho_full = (rho_half_lower.values + rho_half_upper.values) / 2
    
    # 重建DataArray，使用新的level坐标
    # Full level的索引应该保持原始half level的命名方式
    # 如果half level是[31, 32, 33, ..., 90]，则full level应该是[31, 32, 33, ..., 89]
    # 即去掉最后一个half level
    original_half_levels = rho_half.level.values
    full_level_indices = original_half_levels[:-1]  # 取前n-1个half level的索引作为full level索引
    
    rho_full = xr.DataArray(
        rho_full,
        coords={
            'time': rho_half.time,
            'level': full_level_indices,
            'lat': rho_half.lat,
            'lon': rho_half.lon
        },
        dims=['time', 'level', 'lat', 'lon']
    )
    
    # 保留原始属性
    rho_full.attrs = rho_half.attrs.copy()
    rho_full.attrs['long_name'] = 'Density at full levels (interpolated from half levels)'
    rho_full.attrs['interpolation_method'] = 'linear average between adjacent half levels'
    
    print(f"输出数据形状: {rho_full.shape}")
    print(f"输出数据level范围: {rho_full.level.values.min()} - {rho_full.level.values.max()}")
    
    return rho_full


# 对三个实验的数据进行插值
print("=" * 60)
print("开始插值 CNTL 数据...")
print("=" * 60)
rho_cntl_full = interpolate_half_to_full_level(rhor_cntl_half)

print("\n" + "=" * 60)
print("开始插值 P4K 数据...")
print("=" * 60)
rho_p4k_full = interpolate_half_to_full_level(rhor_p4k_half)

print("\n" + "=" * 60)
print("开始插值 4CO2 数据...")
print("=" * 60)
rho_4co2_full = interpolate_half_to_full_level(rhor_4co2_half)

print("\n" + "=" * 60)
print("插值完成！")
print("=" * 60)

开始插值 CNTL 数据...
输入数据形状: (5114, 22, 17, 180)
输入数据level范围: 31 - 90


In [ ]:
rho_cntl_full

<xarray.DataArray (time: 5114, level: 21, lat: 17, lon: 180)> Size: 3GB
array([[[[       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [0.075821  , 0.07625933, 0.07624471, ..., 0.07568635,
          0.07594   , 0.07597771],
         [0.07619131, 0.07574799, 0.0758037 , ..., 0.07557053,
          0.07530398, 0.07573177],
         ...,
         [0.0756121 , 0.07561746, 0.07536426, ..., 0.07544172,
          0.07525568, 0.07546   ],
         [0.07549901, 0.07564616, 0.07564752, ..., 0.07533176,
          0.07544955, 0.07546999],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan]],

        [[       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [0.12027624, 0.12064362, 0.12092046, ..., 0.11988979,
          0.12046294, 0.12022316],
         [0.12007773, 0.1203167 , 0.12044362, ..., 0.11997484,
          0.11988271, 0.1197271 ],
...
         [1.14003269, 1.14166526, 1.15125597, ..., 1.1248466 ,
          1.13045724, 1.13398714],
         [1.1487539 , 1.15557468, 1.15754631, ..., 1.13634831,
          1.13137969, 1.14092946],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan]],

        [[       nan,        nan,        nan, ...,        nan,
                 nan,        nan],
         [1.18814648, 1.18907937, 1.18880308, ..., 1.18715746,
          1.18790322, 1.18802482],
         [1.18571194, 1.18456601, 1.18354674, ..., 1.18437266,
          1.18486317, 1.18494925],
         ...,
         [1.1486898 , 1.15029965, 1.15996841, ..., 1.1346385 ,
          1.13895542, 1.14242217],
         [1.15766503, 1.16454491, 1.16648919, ..., 1.14578418,
          1.13991287, 1.14979854],
         [       nan,        nan,        nan, ...,        nan,
                 nan,        nan]]]], shape=(5114, 21, 17, 180))
Coordinates:
  * time     (time) datetime64[ns] 41kB 1980-01-01 1980-01-02 ... 1993-12-31
  * level    (level) int64 168B 31 35 38 41 46 51 55 58 ... 80 81 83 84 85 87 89
  * lat      (lat) float64 136B -16.0 -14.0 -12.0 -10.0 ... 10.0 12.0 14.0 16.0
  * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
Attributes:
    long_name:             Density at full levels (interpolated from half lev...
    interpolation_method:  linear average between adjacent half levels

In [ ]:
# 保存full level的密度数据
output_dir_cntl = Path('../3D_data/cntl/')
output_dir_p4k = Path('../3D_data/p4k/')
output_dir_4co2 = Path('../3D_data/4co2/')

# 确保输出目录存在
output_dir_cntl.mkdir(parents=True, exist_ok=True)
output_dir_p4k.mkdir(parents=True, exist_ok=True)
output_dir_4co2.mkdir(parents=True, exist_ok=True)

print("保存数据到磁盘...")

# 保存CNTL (使用压缩以节省空间)
output_file_cntl = output_dir_cntl / 'rho_full_levels.nc'
print(f"保存 CNTL: {output_file_cntl}")
rho_cntl_full.to_dataset(name='rho').to_netcdf(output_file_cntl)

# 保存P4K
output_file_p4k = output_dir_p4k / 'rho_full_levels.nc'
print(f"保存 P4K: {output_file_p4k}")
rho_p4k_full.to_dataset(name='rho').to_netcdf(output_file_p4k)

# 保存4CO2
output_file_4co2 = output_dir_4co2 / 'rho_full_levels.nc'
print(f"保存 4CO2: {output_file_4co2}")
rho_4co2_full.to_dataset(name='rho').to_netcdf(output_file_4co2)

print("\n所有数据已成功保存！")
print(f"  - CNTL: {output_file_cntl}")
print(f"  - P4K: {output_file_p4k}")
print(f"  - 4CO2: {output_file_4co2}")

保存数据到磁盘...
保存 CNTL: ../3D_data/cntl/rho_full_levels.nc
保存 P4K: ../3D_data/p4k/rho_full_levels.nc
保存 4CO2: ../3D_data/4co2/rho_full_levels.nc

所有数据已成功保存！
  - CNTL: ../3D_data/cntl/rho_full_levels.nc
  - P4K: ../3D_data/p4k/rho_full_levels.nc
  - 4CO2: ../3D_data/4co2/rho_full_levels.nc


In [ ]:
output_dir_cntl

PosixPath('../3D_data/cntl')

In [ ]:
# 验证插值结果
print("=" * 60)
print("验证插值结果")
print("=" * 60)

# 选择一个时间点和位置进行验证
time_idx = 100
lat_idx = rho_cntl_full.sizes['lat'] // 2
lon_idx = rho_cntl_full.sizes['lon'] // 2

print(f"\n选择验证点: time={time_idx}, lat_idx={lat_idx}, lon_idx={lon_idx}")

# 提取垂直剖面
half_profile = rhor_cntl_half.isel(time=time_idx, lat=lat_idx, lon=lon_idx).values
full_profile = rho_cntl_full.isel(time=time_idx, lat=lat_idx, lon=lon_idx).values

print(f"\nHalf level 层数: {len(half_profile)}")
print(f"Full level 层数: {len(full_profile)}")

# 检查插值关系
print("\n验证插值公式 rho_full[k] = (rho_half[k-1/2] + rho_half[k+1/2]) / 2:")
for k in range(min(5, len(full_profile))):  # 只显示前5层
    expected = (half_profile[k] + half_profile[k+1]) / 2
    actual = full_profile[k]
    diff = abs(expected - actual)
    print(f"  Level {k+1}: rho_full={actual:.6f}, expected={(half_profile[k] + half_profile[k+1])/2:.6f}, diff={diff:.2e}")

# 可视化垂直剖面
fig, ax = plt.subplots(1, 1, figsize=(8, 10))

ax.plot(half_profile, range(len(half_profile)), 'o-', label='Half levels', markersize=4)
ax.plot(full_profile, range(1, len(full_profile)+1), 's-', label='Full levels (interpolated)', markersize=4)

ax.set_ylabel('Level Index')
ax.set_xlabel('Density (kg/m³)')
ax.set_title('Vertical Profile: Half vs Full Levels')
ax.legend()
ax.grid(True, alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../figures/rho_half_vs_full_profile.png', dpi=150, bbox_inches='tight')
print(f"\n图片已保存至: ../figures/rho_half_vs_full_profile.png")
plt.show()

print("\n✓ 验证完成！")

验证插值结果


NameError: name 'rho_cntl_full' is not defined